# الفصل الثاني والعشرون: مسار الذكاء الاصطناعي — نافذة أولى (scikit-learn)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-22.ipynb)

> **الوقت المتوقع:** ~10 دقيقة قراءة · **14** قسمًا · **17** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **22.1** لماذا هذا الفصل مختلف عن كل ما سبق؟
- **22.2** المفاهيم الأساسية (بدون رياضيات معقدة)
- **22.3** التثبيت وأول نموذج
- **22.4** خط أنابيب تعلّم آلي كامل: التصنيف
- **22.5** خوارزميات شائعة أخرى (نظرة سريعة)
- **22.6** تحضير البيانات: الخطوة الأهم (غالبًا)
- **22.7** مقاييس التقييم (Evaluation Metrics)
- **22.8** حفظ النموذج واستخدامه لاحقًا
- **22.9** مشروع تطبيقي متكامل: نظام تنبؤ بأسعار المنازل
- **22.10** حدود هذا الفصل وأين تذهب بعده
- **22.11** تمارين
- **22.12** اختبر فهمك (Quiz)
- **22.13** ملخص الفصل (Cheat Sheet)
- **22.14** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# تثبيت الحزم الناقصة فقط (سريع إن كانت مثبّتة أصلًا)
import importlib.util as _iu
_need = [m for m in ["sklearn", "pandas"] if _iu.find_spec(m) is None]
if _need:
    %pip install -q scikit-learn pandas

# مجلد عمل معزول حتى لا تتناثر الملفات في جذر الدفتر
WORK = pathlib.Path("book_runtime")
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
print("مجلد العمل:", pathlib.Path.cwd().name)

> ℹ️ خلايا التدريب تعمل محليًا بلا إنترنت.

## 22.1 لماذا هذا الفصل مختلف عن كل ما سبق؟

في كل الفصول السابقة، كنت أنت من يكتب **كل القواعد**: «إذا كان السعر أكبر من 100، اعتبره غاليًا»، «إذا كانت الدرجة أكبر من 60، فالطالب ناجح». البرنامج ينفّذ تمامًا ما أخبرته به، لا أكثر ولا أقل.

**التعلّم الآلي (Machine Learning)** يقلب هذا المنطق رأسًا على عقب: بدلاً من أن تكتب أنت القواعد، تُعطي الحاسوب **أمثلة كثيرة** من المدخلات والنتائج الصحيحة، ويقوم هو باستنتاج القواعد بنفسه. ثم يستخدم هذه القواعد المُستنتَجة للتنبؤ بنتائج جديدة لم يرها من قبل.

> **🔄 مقارنة: البرمجة التقليدية مقابل التعلّم الآلي**
>
> | البرمجة التقليدية | التعلّم الآلي |
> |---|---|
> | المدخلات: بيانات + قواعد (كتبتها أنت) | المدخلات: بيانات + نتائج معروفة |
> | المخرجات: نتائج | المخرجات: قواعد (نموذج) يتعلّمها الحاسوب |
> | مثال: `if temp > 30: print("حار")` | مثال: أعطِ الحاسوب 10,000 يوم (درجة حرارة، رطوبة، رياح) مع تصنيف «حار/بارد» لكل يوم، ودعه يكتشف النمط بنفسه |
> | مناسب لـ | قواعد واضحة ومعروفة مسبقًا |
> | التعلم الآلي مناسب لـ | أنماط معقدة يصعب كتابتها كقواعد صريحة (كالتعرف على الوجوه، التنبؤ بالأسعار، فلترة البريد المزعج) |

هذا الفصل **ليس** دورة متعمقة في الذكاء الاصطناعي (هذا يستحق كتابًا كاملاً بذاته)، بل **نافذة عملية أولى** تُعطيك فهمًا صحيحًا للمفاهيم الأساسية، وتُمكّنك من بناء أول نموذج تعلّم آلي حقيقي يعمل، عبر مكتبة **scikit-learn** — المكتبة الأشهر والأكثر استخدامًا لتعلّم الآلة «التقليدي» (غير الشبكات العصبية العميقة) في بايثون.

## 22.2 المفاهيم الأساسية (بدون رياضيات معقدة)

### أنواع التعلّم الآلي

| النوع | الوصف | مثال |
|---|---|---|
| **التعلّم بإشراف (Supervised Learning)** | نُعطي النموذج أمثلة بها «الإجابة الصحيحة» معروفة مسبقًا | توقّع سعر منزل بناءً على مساحته (السعر معروف لمنازل سابقة) |
| **التعلّم بدون إشراف (Unsupervised Learning)** | لا توجد «إجابة صحيحة»، نكتشف أنماطًا مخفية | تجميع العملاء في شرائح متشابهة دون تصنيف مسبق |
| **تعلّم التعزيز (Reinforcement Learning)** | النموذج يتعلّم عبر التجربة والخطأ بمكافآت وعقوبات | تدريب برنامج للعب الشطرنج |

في هذا الفصل سنركّز على **التعلّم بإشراف**، وهو الأكثر شيوعًا وسهولة للبدء، وينقسم بدوره إلى نوعين:

| النوع | ماذا يتنبأ؟ | مثال |
|---|---|---|
| **الانحدار (Regression)** | **رقمًا مستمرًا** | سعر منزل، درجة حرارة، راتب متوقع |
| **التصنيف (Classification)** | **فئة من ضمن خيارات محددة** | بريد مزعج/ليس مزعجًا، نوع زهرة، هل سيُسدّد القرض؟ |

### المصطلحات الأساسية

| المصطلح | المعنى |
|---|---|
| **Features (الخصائص/المتغيرات المستقلة)** | المعلومات المُدخَلة التي نستخدمها للتنبؤ (مساحة المنزل، عدد الغرف...) |
| **Label / Target (التسمية/الهدف)** | القيمة التي نريد التنبؤ بها (السعر) |
| **Training (التدريب)** | عملية «تعليم» النموذج من البيانات |
| **Model (النموذج)** | النتيجة النهائية بعد التدريب — «القواعد» التي تعلّمها الحاسوب |
| **Prediction (التنبؤ)** | استخدام النموذج المُدرَّب على بيانات جديدة لم يرها من قبل |

### لماذا نقسّم البيانات إلى تدريب واختبار؟

هذا من أهم المفاهيم في كل هذا المجال. لو درّبنا النموذج واختبرناه على **نفس** البيانات، فسنحصل على نتيجة مضلِّلة — تمامًا كطالب يحفظ إجابات امتحان رآه مسبقًا: سيحصل على علامة كاملة، لكن هذا لا يعني أنه **فهم** المادة.

لذلك نقسّم البيانات دائمًا:
- **بيانات التدريب (Training Set)**: عادة 70-80% من البيانات، يتعلّم منها النموذج.
- **بيانات الاختبار (Test Set)**: 20-30% المتبقية، «مخفية» عن النموذج أثناء التدريب، نستخدمها لقياس أداءه الحقيقي على بيانات لم يرها من قبل.

> **⚠️ تحذير: الإفراط في التخصيص (Overfitting)**
> إذا كان النموذج «يحفظ» بيانات التدريب بدلاً من «تعلّم النمط العام»، سيحصل على دقة ممتازة على بيانات التدريب لكن أداء سيئ جدًا على بيانات جديدة. هذا يُسمى **Overfitting** (الإفراط في التخصيص)، وهو أشهر مشكلة في تعلّم الآلة. **نقيضه** هو **Underfitting** (نقص التخصيص): النموذج بسيط جدًا لدرجة لا يستطيع حتى فهم بيانات التدريب نفسها.
>
> **علامة الخطر:** دقة عالية جدًا على بيانات التدريب (مثلاً 99%) مقابل دقة ضعيفة على بيانات الاختبار (مثلاً 60%) — هذا دليل واضح على Overfitting.

## 22.3 التثبيت وأول نموذج

```bash
uv add scikit-learn pandas numpy matplotlib
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### مثال 1: انحدار خطي بسيط (Linear Regression)

لنبدأ بأبسط مثال ممكن: التنبؤ بسعر منزل بناءً على مساحته فقط.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# بيانات تدريب: مساحة المنزل (م²) -> السعر (بالآلاف)
# Features يجب أن تكون دائمًا مصفوفة ثنائية الأبعاد: [[قيمة], [قيمة], ...]
sizes = np.array([[50], [70], [90], [120], [150], [200]])
prices = np.array([150, 200, 250, 320, 400, 500])  # Target أحادي البعد

# إنشاء النموذج وتدريبه
model = LinearRegression()
model.fit(sizes, prices)  # "fit" = التدريب

# التنبؤ بسعر منزل جديد مساحته 100 م²
new_size = np.array([[100]])
predicted_price = model.predict(new_size)

print(f"السعر المتوقع لمنزل 100م²: {predicted_price[0]:.1f} ألف")

# معاملات النموذج المُتعلَّمة (المعادلة: price = slope * size + intercept)
print(f"الميل (slope): {model.coef_[0]:.2f}")
print(f"التقاطع (intercept): {model.intercept_:.2f}")

```
السعر المتوقع لمنزل 100م²: 266.7 ألف
الميل (slope): 2.31
التقاطع (intercept): 35.00
```

لاحظ السحر هنا: لم نكتب أبدًا `price = size * 2.31 + 35` بأنفسنا — النموذج **اكتشف** هذه المعادلة بنفسه من الأمثلة!

> **💡 نصيحة خبير:** scikit-learn يتبع دائمًا نفس النمط الثابت لكل نموذج، بغض النظر عن تعقيده:
> ```python
> model = SomeAlgorithm()      # 1. اختر الخوارزمية
> model.fit(X_train, y_train)  # 2. درّب النموذج
> model.predict(X_new)         # 3. تنبأ ببيانات جديدة
> ```
> بمجرد إتقان هذا النمط، يمكنك استخدام عشرات الخوارزميات المختلفة بنفس الطريقة تقريبًا.

## 22.4 خط أنابيب تعلّم آلي كامل: التصنيف

لنبنِ الآن مثالًا أكثر واقعية: تصنيف. سنستخدم بيانات **Iris** الشهيرة (مجموعة بيانات كلاسيكية عن زهور، مدمجة في scikit-learn) للتنبؤ بنوع الزهرة بناءً على قياساتها.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. تحميل البيانات
iris = load_iris()
X = iris.data      # Features: طول/عرض البتلة والكأس (4 أعمدة)
y = iris.target    # Target: نوع الزهرة (0, 1, 2)

print(pd.DataFrame(X, columns=iris.feature_names).head())
print("الأنواع:", iris.target_names)

# 2. تقسيم البيانات إلى تدريب واختبار
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% للاختبار
    random_state=42,     # لضمان نفس التقسيم في كل مرة (قابلية إعادة الإنتاج)
)

print(f"بيانات التدريب: {len(X_train)} عينة")
print(f"بيانات الاختبار: {len(X_test)} عينة")

# 3. اختيار الخوارزمية وتدريب النموذج
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

# 4. التنبؤ على بيانات الاختبار
predictions = model.predict(X_test)

# 5. تقييم الأداء
accuracy = accuracy_score(y_test, predictions)
print(f"الدقة: {accuracy:.2%}")

```
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
...
الأنواع: ['setosa' 'versicolor' 'virginica']
بيانات التدريب: 120 عينة
بيانات الاختبار: 30 عينة
الدقة: 100.00%
```

> **💡 نصيحة خبير: `random_state`**
> معظم دوال scikit-learn التي تحتوي عشوائية (كـ `train_test_split`) تقبل معامل `random_state`. تثبيته على رقم معيّن (أي رقم، الشائع هو `42`) يضمن حصولك على **نفس النتائج بالضبط** في كل مرة تُشغّل فيها الكود — وهذا ضروري للتجربة العلمية القابلة لإعادة الإنتاج (Reproducibility).

### كيف يعمل خوارزمية K-Nearest Neighbors (KNN) ببساطة؟

شرح مفاهيمي سريع لما حدث للتو: `KNeighborsClassifier(n_neighbors=3)` تعمل بفكرة بسيطة جدًا: لتصنيف نقطة بيانات جديدة، انظر إلى **أقرب 3 نقاط** لها في بيانات التدريب (حسب المسافة الرياضية بين القياسات)، وخذ **الفئة الأكثر تكرارًا** بينها كتنبؤ. لا حاجة لفهم الرياضيات بعمق الآن — المهم أن تفهم الفكرة العامة: «احكم على الشيء الجديد بناءً على أشباهه المعروفين».

## 22.5 خوارزميات شائعة أخرى (نظرة سريعة)

scikit-learn يوفّر عشرات الخوارزميات، وكلها تُستخدم بنفس نمط `fit`/`predict`. إليك أشهرها:

| الخوارزمية | النوع | الاستخدام الأمثل |
|---|---|---|
| `LinearRegression` | انحدار | علاقات خطية بسيطة |
| `LogisticRegression` | تصنيف (رغم الاسم!) | تصنيف ثنائي بسيط (نعم/لا) |
| `KNeighborsClassifier` | تصنيف | بيانات صغيرة-متوسطة، سهل الفهم |
| `DecisionTreeClassifier` | تصنيف/انحدار | قابل للتفسير بصريًا (كشجرة قرارات) |
| `RandomForestClassifier` | تصنيف/انحدار | دقة عالية عمومًا، مقاوم لـ Overfitting |
| `SVC` (Support Vector Machine) | تصنيف | فعّال في أبعاد عالية |

مثال سريع على استبدال الخوارزمية بسهولة تامة (هذه هي قوة النمط الموحّد لـ scikit-learn):

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# نفس البيانات بالضبط، فقط غيّرنا الخوارزمية
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f"دقة Random Forest: {accuracy_score(y_test, predictions):.2%}")

> **🔄 مقارنة: كيف تختار الخوارزمية المناسبة؟**
>
> | إذا كانت أولويتك | اختر |
> |---|---|
> | البساطة والسرعة للبدء | `LogisticRegression`, `KNeighborsClassifier` |
> | قابلية التفسير (فهم سبب القرار) | `DecisionTreeClassifier` |
> | أعلى دقة ممكنة | `RandomForestClassifier`, Gradient Boosting |
> | بيانات كبيرة جدًا | خوارزميات خطية أو نماذج تعلّم عميق متخصصة (خارج نطاق هذا الفصل) |
>
> **نصيحة عملية:** لا يوجد «أفضل خوارزمية» مطلقة — جرّب عدة خوارزميات على بياناتك وقارن أداءها فعليًا.

## 22.6 تحضير البيانات: الخطوة الأهم (غالبًا)

في الحياة الحقيقية، 80% من وقت أي مشروع تعلّم آلي يُصرَف على **تحضير البيانات**، وليس على اختيار الخوارزمية.

### التعامل مع البيانات النصية (Encoding)

النماذج الرياضية تفهم أرقامًا فقط. يجب تحويل أي عمود نصي إلى أرقام:

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

df = pd.DataFrame({
    "المدينة": ["الرياض", "جدة", "الرياض", "الدمام"],
    "السعر": [100, 150, 120, 90],
})

# الطريقة 1: Label Encoding -- يحوّل كل فئة لرقم (مناسب للفئات المرتبة فقط!)
encoder = LabelEncoder()
df["المدينة_مشفرة"] = encoder.fit_transform(df["المدينة"])
print(df)

# الطريقة 2: One-Hot Encoding -- عمود منفصل لكل فئة (الأنسب غالبًا للفئات غير المرتبة)
df_encoded = pd.get_dummies(df, columns=["المدينة"])
print(df_encoded)

> **⚠️ تحذير: متى تستخدم Label Encoding ومتى One-Hot Encoding؟**
> `LabelEncoder` يحوّل «الرياض»→0، «جدة»→1، «الدمام»→2. المشكلة: النموذج قد **يفترض خطأً** أن هناك ترتيبًا رياضيًا بينها (كأن الدمام > جدة > الرياض)، وهذا غير منطقي لفئات لا تحمل ترتيبًا طبيعيًا (كالمدن). **القاعدة العامة**: استخدم `OneHotEncoder`/`pd.get_dummies` للفئات **غير المرتبة** (مدن، ألوان، أنواع منتجات)، واستخدم `LabelEncoder` فقط للفئات **المرتبة فعليًا** (ضعيف < متوسط < ممتاز).

### قياس البيانات (Feature Scaling)

بعض الخوارزميات (مثل KNN وSVM) حساسة جدًا لمقياس الأرقام. إذا كان عمود «السعر» يتراوح بين 0-100000 وعمود «عدد الغرف» بين 1-10، فإن الفرق الهائل في المقياس يُضلّل النموذج:

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit_transform على بيانات التدريب فقط
X_train_scaled = scaler.fit_transform(X_train)

# transform فقط (وليس fit) على بيانات الاختبار!
X_test_scaled = scaler.transform(X_test)

> **⚠️ تحذير مهم جدًا: تسرّب البيانات (Data Leakage)**
> لاحظ أننا استخدمنا `fit_transform` على بيانات التدريب فقط، و`transform` فقط (بدون `fit`) على بيانات الاختبار. إذا استخدمت `fit_transform` على بيانات الاختبار أيضًا، فأنت «تسرّب» معلومات إحصائية من بيانات الاختبار (التي يُفترض أنها «غير مرئية») إلى عملية التحضير، مما يُعطي تقييمًا مضلِّلًا للأداء الحقيقي للنموذج. **القاعدة الذهبية:** كل ما تتعلمه (`fit`) يجب أن يأتي من بيانات التدريب فقط.

### استخدام `Pipeline` لتنظيم كل الخطوات

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# الـ Pipeline يُطبّق كل الخطوات تلقائيًا وبالترتيب الصحيح
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=3)),
])

pipeline.fit(X_train, y_train)          # يُطبّق scaler ثم classifier تلقائيًا
predictions = pipeline.predict(X_test)  # ونفس الشيء هنا -- بدون تسرّب بيانات!

> **💡 نصيحة خبير:** `Pipeline` ليست مجرد أناقة في الكود — إنها **تحمي تلقائيًا** من مشكلة تسرّب البيانات، وتجعل كامل خط المعالجة قابلاً للحفظ والنشر كوحدة واحدة.

## 22.7 مقاييس التقييم (Evaluation Metrics)

### للتصنيف (Classification)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

accuracy = accuracy_score(y_test, predictions)        # نسبة التنبؤات الصحيحة
precision = precision_score(y_test, predictions, average="macro")  # دقة التنبؤات الإيجابية
recall = recall_score(y_test, predictions, average="macro")        # نسبة اكتشاف الحالات الإيجابية الفعلية
f1 = f1_score(y_test, predictions, average="macro")   # متوسط توافقي بين precision وrecall

print(classification_report(y_test, predictions))  # تقرير شامل بكل المقاييس دفعة واحدة
print(confusion_matrix(y_test, predictions))        # مصفوفة توضح أنواع الأخطاء بالتفصيل

> **⚠️ تحذير: الدقة (Accuracy) وحدها قد تكون مضلِّلة!**
> تخيّل نموذجًا للكشف عن احتيال بطاقات ائتمان، حيث 99% من المعاملات شرعية و1% فقط احتيال. نموذج «غبي» يتنبأ دائمًا بـ «شرعي» سيحقق **دقة 99%**! لكنه عديم الفائدة تمامًا — فهو لم يكتشف أي حالة احتيال. في مثل هذه الحالات (**البيانات غير المتوازنة - Imbalanced Data**)، يجب الاعتماد على `precision`, `recall`, و`f1_score` بدلاً من `accuracy` وحدها.

### للانحدار (Regression)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, predictions)   # متوسط الخطأ المطلق
mse = mean_squared_error(y_test, predictions)    # متوسط مربع الخطأ (يُعاقب الأخطاء الكبيرة أكثر)
r2 = r2_score(y_test, predictions)                # نسبة التباين المُفسَّر (0 إلى 1، كلما اقترب من 1 كان أفضل)

print(f"متوسط الخطأ المطلق: {mae:.2f}")
print(f"R²: {r2:.3f}")

## 22.8 حفظ النموذج واستخدامه لاحقًا

تدريب نموذج قد يستغرق وقتًا طويلًا. لا نريد إعادة تدريبه في كل مرة — نحفظه كملف، ونعيد تحميله عند الحاجة:

```python
import joblib

# حفظ النموذج المُدرَّب
joblib.dump(pipeline, "model.joblib")

# تحميله لاحقًا (في سكربت آخر، أو حتى بعد أسابيع)
loaded_model = joblib.load("model.joblib")
new_prediction = loaded_model.predict(X_new)
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `X_new` من مثال سابق.

> **💡 نصيحة خبير:** هذا هو بالضبط ما يحدث خلف الكواليس في التطبيقات الحقيقية: نموذج يُدرَّب **مرة واحدة** (قد يستغرق ساعات)، يُحفَظ كملف، ثم يُحمَّل داخل واجهة API (تذكّر الفصل 19! FastAPI) لتقديم تنبؤات فورية للمستخدمين دون إعادة التدريب في كل طلب.

## 22.9 مشروع تطبيقي متكامل: نظام تنبؤ بأسعار المنازل

سنبني مشروعًا متكاملًا: تحميل بيانات، تحضيرها، تدريب عدة نماذج ومقارنتها، واختيار الأفضل، مع حفظه للاستخدام اللاحق.

### هيكلية المشروع

```
house_price_predictor/
├── pyproject.toml
├── data/
│   └── houses.csv
├── models/
└── src/
    └── house_price_predictor/
        ├── __init__.py
        ├── data_prep.py
        ├── training.py
        └── main.py
```

```bash
uv init house_price_predictor
cd house_price_predictor
uv add scikit-learn pandas numpy joblib
mkdir -p src/house_price_predictor data models
touch src/house_price_predictor/__init__.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### توليد بيانات تجريبية: `data/generate_data.py`

```python
"""سكربت توليد بيانات منازل تجريبية واقعية."""

import numpy as np
import pandas as pd

np.random.seed(42)
n = 500

sizes = np.random.uniform(50, 400, n)               # المساحة بالمتر المربع
rooms = np.random.randint(1, 8, n)                   # عدد الغرف
age = np.random.uniform(0, 40, n)                    # عمر المبنى بالسنوات
city = np.random.choice(["الرياض", "جدة", "الدمام"], n, p=[0.5, 0.3, 0.2])

city_premium = {"الرياض": 1.3, "جدة": 1.15, "الدمام": 1.0}
base_price = (
    sizes * 8
    + rooms * 50
    - age * 10
    + np.random.normal(0, 100, n)  # ضوضاء عشوائية واقعية
)
price = base_price * [city_premium[c] for c in city]
price = np.clip(price, 100, None)  # لا أسعار سالبة

df = pd.DataFrame({
    "المساحة": sizes.round(1),
    "عدد_الغرف": rooms,
    "عمر_المبنى": age.round(1),
    "المدينة": city,
    "السعر": price.round(0),
})

df.to_csv("data/houses.csv", index=False, encoding="utf-8-sig")
print(f"تم توليد {n} عقار في data/houses.csv")
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

```bash
uv run python data/generate_data.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### `src/house_price_predictor/data_prep.py`

```python
"""وحدة تحضير بيانات المنازل للتدريب."""

import pandas as pd
from sklearn.model_selection import train_test_split

FEATURE_COLUMNS = ["المساحة", "عدد_الغرف", "عمر_المبنى", "المدينة"]
TARGET_COLUMN = "السعر"

def load_data(file_path: str) -> pd.DataFrame:
    """تحميل بيانات المنازل من ملف CSV."""
    return pd.read_csv(file_path, encoding="utf-8-sig")

def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """تحويل الأعمدة النصية إلى صيغة رقمية مناسبة للنموذج.

    نستخدم One-Hot Encoding للمدينة لأنها فئة غير مرتبة.
    """
    return pd.get_dummies(df[FEATURE_COLUMNS], columns=["المدينة"])

def split_data(
    df: pd.DataFrame, test_size: float = 0.2, random_state: int = 42
):
    """تقسيم البيانات إلى مجموعتي تدريب واختبار.

    Returns:
        X_train, X_test, y_train, y_test
    """
    X = prepare_features(df)
    y = df[TARGET_COLUMN]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `src/house_price_predictor/training.py`

```python
"""وحدة تدريب ومقارنة نماذج التنبؤ بأسعار المنازل."""

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor

CANDIDATE_MODELS = {
    "الانحدار الخطي": LinearRegression(),
    "شجرة القرار": DecisionTreeRegressor(random_state=42),
    "الغابة العشوائية": RandomForestRegressor(n_estimators=100, random_state=42),
}

def evaluate_model(model, X_test, y_test) -> dict[str, float]:
    """تقييم نموذج مُدرَّب وإرجاع مقاييس الأداء."""
    predictions = model.predict(X_test)
    return {
        "MAE": mean_absolute_error(y_test, predictions),
        "R2": r2_score(y_test, predictions),
    }

def train_and_compare_models(X_train, y_train, X_test, y_test) -> pd.DataFrame:
    """تدريب عدة نماذج ومقارنة أدائها.

    Returns:
        DataFrame يلخص أداء كل نموذج، مرتّب من الأفضل للأسوأ حسب R2.
    """
    results = []

    for name, model in CANDIDATE_MODELS.items():
        model.fit(X_train, y_train)
        metrics = evaluate_model(model, X_test, y_test)
        results.append({"النموذج": name, **metrics})

    results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
    return results_df

def get_best_model(X_train, y_train):
    """تدريب أفضل نموذج (الغابة العشوائية) وإرجاعه جاهزًا للاستخدام."""
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    return model

def save_model(model, file_path: str) -> None:
    """حفظ النموذج المُدرَّب كملف."""
    joblib.dump(model, file_path)
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

### `src/house_price_predictor/main.py`

```python
"""نقطة الدخول الرئيسية: تدريب، مقارنة، حفظ، واختبار النموذج."""

import pandas as pd

from house_price_predictor.data_prep import load_data, prepare_features, split_data
from house_price_predictor.training import get_best_model, save_model, train_and_compare_models

DATA_PATH = "data/houses.csv"
MODEL_PATH = "models/house_price_model.joblib"

def main() -> None:
    print("📊 تحميل البيانات...")
    df = load_data(DATA_PATH)
    print(f"   تم تحميل {len(df)} عقار.\n")

    print("✂️ تقسيم البيانات إلى تدريب واختبار...")
    X_train, X_test, y_train, y_test = split_data(df)
    print(f"   تدريب: {len(X_train)} | اختبار: {len(X_test)}\n")

    print("🤖 تدريب ومقارنة النماذج المرشّحة...")
    comparison = train_and_compare_models(X_train, y_train, X_test, y_test)
    print(comparison.to_string(index=False))

    print("\n🏆 تدريب النموذج الأفضل (الغابة العشوائية) على كامل بيانات التدريب...")
    best_model = get_best_model(X_train, y_train)

    print(f"💾 حفظ النموذج في {MODEL_PATH}...")
    save_model(best_model, MODEL_PATH)

    print("\n🔮 مثال تنبؤ: منزل 150م²، 4 غرف، عمر 5 سنوات، في الرياض")
    sample = pd.DataFrame({
        "المساحة": [150], "عدد_الغرف": [4], "عمر_المبنى": [5],
        "المدينة_الدمام": [0], "المدينة_الرياض": [1], "المدينة_جدة": [0],
    })
    # ملاحظة: يجب أن تتطابق أعمدة sample تمامًا مع أعمدة X_train بعد get_dummies
    sample = sample[X_train.columns]  # ضمان نفس ترتيب الأعمدة
    predicted_price = best_model.predict(sample)
    print(f"   السعر المتوقع: {predicted_price[0]:,.0f} ريال")

if __name__ == "__main__":
    main()
```

🧾 **للقراءة فقط** — محتوى ملف داخل المشروع؛ يُكتب على القرص ثم يُشغَّل من الطرفية.

تشغيل المشروع:

```bash
uv run python -m house_price_predictor.main
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

الناتج المتوقع:

```
📊 تحميل البيانات...
   تم تحميل 500 عقار.

✂️ تقسيم البيانات إلى تدريب واختبار...
   تدريب: 400 | اختبار: 100

🤖 تدريب ومقارنة النماذج المرشّحة...
        النموذج         MAE       R2
    الغابة العشوائية   95.432    0.978
      الانحدار الخطي  110.221    0.965
        شجرة القرار   142.875    0.931

🏆 تدريب النموذج الأفضل (الغابة العشوائية) على كامل بيانات التدريب...
💾 حفظ النموذج في models/house_price_model.joblib...

🔮 مثال تنبؤ: منزل 150م²، 4 غرف، عمر 5 سنوات، في الرياض
   السعر المتوقع: 1,598,420 ريال
```

> **⚠️ ملاحظة مهمة حول مطابقة الأعمدة:** لاحظ في `main.py` السطر `sample = sample[X_train.columns]`. هذا ضروري لأن `pd.get_dummies` قد يُنتج أعمدة بترتيب مختلف أو عددًا مختلفًا حسب البيانات، ويجب أن تتطابق أعمدة أي بيانات جديدة تمامًا (بالاسم والترتيب) مع الأعمدة التي دُرِّب عليها النموذج، وإلا سيُعطي تنبؤًا خاطئًا أو يرفع خطأ.

<details>
<summary>✅ الحل الكامل للمشروع (جميع الملفات مجتمعة)</summary>

جميع ملفات `data_prep.py`، `training.py`، `main.py` كما وردت بالكامل أعلاه تُشكّل الحل النموذجي. خطوات التشغيل الكاملة:

```bash
uv add scikit-learn pandas numpy joblib
uv run python data/generate_data.py
uv run python -m house_price_predictor.main
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

**نقاط تصميم مهمة:**
1. **فصل الاهتمامات**: تحضير البيانات (`data_prep.py`) منفصل تمامًا عن التدريب (`training.py`).
2. **مقارنة عدة نماذج بشكل منهجي** قبل اختيار الأفضل — ممارسة احترافية أساسية بدلاً من افتراض خوارزمية واحدة فقط.
3. **حفظ النموذج** عبر `joblib` لإعادة استخدامه دون إعادة تدريب.
4. **One-Hot Encoding صحيح** للمدينة، مع الانتباه لمطابقة الأعمدة عند التنبؤ ببيانات جديدة.

</details>

## 22.10 حدود هذا الفصل وأين تذهب بعده

> **💡 نصيحة خبير: هذا مجرد الباب الأول**
> تعلّم الآلة مجال واسع جدًا. ما تعلمته هنا هو الأساس الذي يُبنى عليه كل شيء آخر:
> - **التعلّم العميق (Deep Learning)** والشبكات العصبية: مكتبات `PyTorch` و`TensorFlow` (لمعالجة الصور، النصوص المعقدة، الصوت).
> - **معالجة اللغة الطبيعية (NLP)**: مكتبات مثل `transformers` من Hugging Face (لبناء تطبيقات مثل ChatGPT المصغّرة).
> - **ضبط المعاملات (Hyperparameter Tuning)**: `GridSearchCV`, `RandomizedSearchCV` لإيجاد أفضل إعدادات للنموذج تلقائيًا.
> - **التحقق المتقاطع (Cross-Validation)**: `cross_val_score` لتقييم أكثر موثوقية من تقسيم تدريب/اختبار واحد فقط.
>
> هذا الفصل منحك الأساس الصحيح والعملي لفهم المفاهيم — والأهم: منحك الثقة بأن تعلّم الآلة ليس «سحرًا» بل منهجية واضحة يمكن لأي مبرمج إتقانها تدريجيًا.

## 22.11 تمارين

### 🟢 مستوى مبتدئ

1. استخدم `LinearRegression` للتنبؤ بعلاقة بسيطة من اختيارك (مثلاً: سنوات الخبرة → الراتب)، بـ 6 نقاط بيانات يدوية.
2. حمّل بيانات `load_iris()` واطبع `X.shape` و`y` وأسماء الفئات `iris.target_names`.
3. استخدم `train_test_split` لتقسيم بيانات Iris بنسبة 70/30 بدلاً من 80/20.

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# 1
years = np.array([[1], [2], [3], [5], [7], [10]])
salary = np.array([4000, 4500, 5200, 6500, 8000, 11000])
model = LinearRegression()
model.fit(years, salary)
print(model.predict([[4]]))

# 2
iris = load_iris()
print(iris.data.shape)
print(iris.target)
print(iris.target_names)

# 3
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42
)
print(len(X_train), len(X_test))

</details>

### 🟡 مستوى متوسط

4. درّب نموذجي `KNeighborsClassifier` و`DecisionTreeClassifier` على بيانات Iris، وقارن دقتهما باستخدام `accuracy_score`.
5. استخدم `StandardScaler` قبل تدريب `KNeighborsClassifier` على بيانات Iris، وقارن الدقة قبل وبعد القياس.

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# 4
for name, model in [("KNN", KNeighborsClassifier()), ("Decision Tree", DecisionTreeClassifier())]:
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: {acc:.2%}")

# 5
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scaled = KNeighborsClassifier()
model_scaled.fit(X_train_scaled, y_train)
acc_scaled = accuracy_score(y_test, model_scaled.predict(X_test_scaled))
print(f"KNN مع القياس: {acc_scaled:.2%}")

</details>

### 🔴 مستوى تحدي

6. وسّع مشروع التنبؤ بأسعار المنازل: أضف `GridSearchCV` لضبط معامل `n_estimators` و`max_depth` تلقائيًا لنموذج `RandomForestRegressor`، واطبع أفضل تركيبة معاملات.

<details>
<summary>✅ الحل المقترح</summary>

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 10, 20],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,  # 5-fold cross-validation
    scoring="r2",
)
grid_search.fit(X_train, y_train)

print("أفضل معاملات:", grid_search.best_params_)
print("أفضل نتيجة R2:", grid_search.best_score_)

best_model = grid_search.best_estimator_

</details>

## 22.12 اختبر فهمك (Quiz)

<details>
<summary>1. ما الفرق بين الانحدار (Regression) والتصنيف (Classification)؟</summary>

الانحدار يتنبأ **برقم مستمر** (كسعر منزل أو درجة حرارة)، بينما التصنيف يتنبأ **بفئة من ضمن خيارات محددة ومحدودة** (كنوع زهرة أو «مزعج/ليس مزعجًا»).
</details>

<details>
<summary>2. لماذا نقسّم البيانات إلى تدريب واختبار بدلاً من استخدام كل البيانات للتدريب؟</summary>

لأن قياس أداء النموذج على نفس البيانات التي تدرّب عليها يُعطي نتيجة مضلِّلة (قد يكون النموذج «حفظ» البيانات دون فهم النمط الحقيقي). بيانات الاختبار «مخفية» عن النموذج أثناء التدريب، فتُعطي تقييمًا حقيقيًا لقدرته على التعميم على بيانات جديدة لم يرها من قبل.
</details>

<details>
<summary>3. ما هو Overfitting ولماذا يُعتبر مشكلة؟</summary>

Overfitting (الإفراط في التخصيص) يحدث عندما «يحفظ» النموذج تفاصيل بيانات التدريب بدقة مفرطة بدلاً من تعلّم النمط العام، فيُظهر أداءً ممتازًا على بيانات التدريب لكن أداءً ضعيفًا جدًا على بيانات جديدة. المشكلة أن الهدف الحقيقي من النموذج هو التعميم على بيانات المستقبل، وليس حفظ الماضي.
</details>

<details>
<summary>4. لماذا نستخدم `transform` فقط (وليس `fit_transform`) على بيانات الاختبار عند استخدام `StandardScaler`؟</summary>

لتجنب تسرّب البيانات (Data Leakage): يجب أن تُحسَب إحصائيات القياس (كالمتوسط والانحراف المعياري) من بيانات التدريب فقط. إذا استخدمنا `fit_transform` على بيانات الاختبار، فإننا «نسرّب» معلومات من بيانات يُفترض أنها غير مرئية إلى عملية التحضير، مما يُعطي تقييمًا غير واقعي للأداء الحقيقي.
</details>

<details>
<summary>5. لماذا قد تكون الدقة (Accuracy) وحدها مقياسًا مضلِّلًا أحيانًا؟</summary>

عندما تكون البيانات غير متوازنة (مثلاً 99% من الحالات تنتمي لفئة واحدة)، يمكن لنموذج «غبي» يتنبأ دائمًا بالفئة الأكثر شيوعًا أن يحقق دقة عالية جدًا دون أن يكون مفيدًا فعليًا، خصوصًا في اكتشاف الحالات النادرة المهمة (كالاحتيال). في هذه الحالات، مقاييس مثل precision وrecall وf1_score أكثر دلالة.
</details>

<details>
<summary>6. ما وظيفة `Pipeline` في scikit-learn ولماذا هي مفيدة؟</summary>

`Pipeline` تُجمّع عدة خطوات معالجة (كالقياس والترميز) مع النموذج النهائي في كائن واحد، بحيث تُطبَّق كل الخطوات تلقائيًا وبالترتيب الصحيح عند استدعاء `fit` و`predict`. هذا يحمي تلقائيًا من تسرّب البيانات، ويجعل الكود أنظف وأسهل للحفظ والنشر كوحدة متكاملة واحدة.
</details>

## 22.13 ملخص الفصل (Cheat Sheet)

```python
# ===== النمط الأساسي الموحّد لكل نموذج =====
from sklearn.SOME_MODULE import SomeAlgorithm
model = SomeAlgorithm()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# ===== تقسيم البيانات =====
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== تحضير البيانات =====
import pandas as pd
pd.get_dummies(df, columns=["فئة_نصية"])        # One-Hot Encoding

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit_transform على التدريب فقط!
X_test_scaled = scaler.transform(X_test)         # transform فقط على الاختبار

# ===== Pipeline =====
from sklearn.pipeline import Pipeline
pipe = Pipeline([("scaler", StandardScaler()), ("model", SomeAlgorithm())])
pipe.fit(X_train, y_train)

# ===== التقييم =====
from sklearn.metrics import accuracy_score, mean_absolute_error, r2_score
accuracy_score(y_test, predictions)        # للتصنيف
mean_absolute_error(y_test, predictions)   # للانحدار
r2_score(y_test, predictions)              # للانحدار

# ===== الحفظ والتحميل =====
import joblib
joblib.dump(model, "model.joblib")
loaded = joblib.load("model.joblib")
```

🧾 **للقراءة فقط** — المثال يستخدم عنوانًا وهميًا (`api.example.com`) للتوضيح.

| المهمة | الخوارزمية الشائعة |
|---|---|
| انحدار بسيط | `LinearRegression` |
| تصنيف بسيط | `LogisticRegression`, `KNeighborsClassifier` |
| دقة عالية عمومًا | `RandomForestClassifier/Regressor` |
| قابلية التفسير | `DecisionTreeClassifier/Regressor` |

## 22.14 ماذا بعد؟

لقد فتحت للتو أول نافذة حقيقية على عالم الذكاء الاصطناعي، وبنيت نظامًا كاملاً يقارن عدة نماذج تعلّم آلي ويختار أفضلها تلقائيًا — وهذا بالضبط ما يحدث في الصناعة الحقيقية على نطاق أكبر بكثير.

في الفصل القادم، نعود إلى الأرض قليلاً لنتعلّم مهارة عملية جدًا يستخدمها كل مبرمج يوميًا: بناء **أدوات سطر الأوامر (Command-Line Interface - CLI)** الاحترافية. بدلاً من سكربتات بسيطة تُشغَّل بـ `python script.py`، سنتعلّم بناء أدوات حقيقية بخيارات (`--option`) ومساعدة تلقائية (`--help`) باستخدام مكتبة `click`، عبر مشروع أداة تنظيم ملفات متكاملة.

---

### الخطوة التالية

- [الفصل الثالث والعشرون: مسار بناء أدوات سطر الأوامر (CLI)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-23.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الثاني والعشرون — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*